# 10 — Volatility window diagnostics (development-only)

Test whether realized volatility over 5, 10, and 20 bars improves next-bar classification. Five chronological expanding folds; newest 20% is **locked**. All variants use identical rows, train-only transforms and fixed training-only return-tertile targets. This experiment **does not** evaluate alternative target definitions or the final holdout.

In [ ]:
from pathlib import Path
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.volatility_diagnostics import (
    run_volatility_diagnostics, summary_volatility, plot_volatility, volatility_schemes
)
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').is_file()),None)
if ROOT is None: raise RuntimeError('Open notebook from within stock_ml_dynamic repository')
INTERVAL='1d'
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates=[ROOT/'artifacts'/'master_tables'/INTERVAL,ROOT/'notebooks'/'artifacts'/'master_tables'/INTERVAL]
MASTER=next((p for p in candidates if all((p/f).is_file() for f in required)),None)
if MASTER is None: raise FileNotFoundError(f'Master artifacts missing. Run Notebook 03. Searched: {candidates}')
master,latest,bundle,manifest=load_master(MASTER)
print('Master directory:',MASTER,'Rows:',len(master),'Symbols:',master.symbol.nunique())

## Configuration

Default: 8 feature configurations × 3 simple classifiers × 5 folds = **120 fits**. For a quicker first pass, set `MODELS` to `("prior_baseline", "logistic_l2")` (80 fits). L1 uses a higher iteration budget than the previous experiment; review convergence warnings.

In [ ]:
N_FOLDS=5
HOLDOUT_FRACTION=.20
MAX_TRAIN_ROWS=120_000
MODELS=('prior_baseline','logistic_l2','logistic_l1')
OUTPUT=ROOT/'artifacts'/'volatility_diagnostics'/INTERVAL
scores,curves,regimes,prevalence,eligibility,dev_boundary=run_volatility_diagnostics(
    master,n_folds=N_FOLDS,holdout_fraction=HOLDOUT_FRACTION,
    max_train_rows=MAX_TRAIN_ROWS,models=MODELS,output_dir=OUTPUT)
print('Development boundary:',dev_boundary)
print('Final holdout: NOT EVALUATED')
display(eligibility.loc[~eligibility.eligible])
display(scores.tail())

## Compare volatility windows and baseline

Interpret small differences cautiously: all configurations share the same validation folds, and we are using those folds for feature selection. The untouched holdout remains reserved for later confirmation.

In [ ]:
summary=plot_volatility(scores,curves,regimes,OUTPUT)
display(summary.round(5))
paired=scores.pivot_table(index=['fold','model'],columns='scheme',values=['accuracy','roc_auc_ovr_macro'])
for metric in ('accuracy','roc_auc_ovr_macro'):
    for scheme in ('no_volatility','vol_5_only','vol_10_only','vol_20_only'):
        paired[(metric,'all_minus_'+scheme)]=paired[(metric,'all_volatility')]-paired[(metric,scheme)]
display(paired.round(5))

## Volatility regimes and class distribution

Low, medium and high volatility groups use the **training fold’s** 20-bar volatility tertiles. This can reveal whether a classifier performs well only during high-volatility periods. It is a diagnostic, not a trading backtest.

In [ ]:
display(regimes.groupby(['model','scheme','volatility_regime'],as_index=False).agg(
    accuracy=('accuracy','mean'),auc=('roc_auc_ovr_macro','mean'),n=('observations','sum')).round(5))
display(prevalence.pivot(index='fold',columns='class_id',values=['train_share','test_share']).round(4))
print('Saved to:',OUTPUT)

## Interpretation checklist

1. Do shorter volatility windows outperform 20-bar volatility consistently in all five folds?
2. Does removing volatility reduce ROC-AUC and balanced accuracy as well as ordinary accuracy?
3. Is the improvement concentrated in one volatility regime or spread across low, medium, and high?
4. Are L1 coefficients converged? If not, treat its ranking cautiously.
5. **Do not evaluate the final 20% holdout in this notebook.**

A separate future experiment can compare volatility-scaled target definitions, but accuracy across differently defined targets is not directly comparable.